In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

In [ ]:
# Import CGM data from Fran
cgm = pd.read_sas(
    "/Users/tim/Library/CloudStorage/OneDrive-UW/UWMDI/Andrea Steck/Advanced CGM and OGTT/Data_Raw/Final data20250801/tim_allcgmclean_gt4days.sas7bdat",
    format="sas7bdat",
    encoding="latin-1",
)
cgm["DateTime"] = pd.to_datetime(
    cgm.newsensortime, unit="s", origin=pd.Timestamp("1960-01-01"), utc=True
).dt.tz_convert("America/Denver")

In [ ]:
def plot_cgm_peaks(
    id,
    df=cgm,
    sensor_time="DateTime",
    glucose="sensor_glucose",
    min_height=100,
    rise=50,
    time_between_peaks=120,
    min_time_of_peak=30,
    smooth=True,
    clip=True,
):
    df = df[df.ID == id]
    dovs = sorted(df["dov_CGM"].unique())
    fig, axes = plt.subplots(len(dovs), 1, figsize=(12, 6 * len(dovs)), sharey=True)
    if len(dovs) == 1:
        axes = [axes]  # ensure iterable
    all_peak_dfs = []
    for ax, dov in zip(axes, dovs):
        dov_df = df[df.dov_CGM == dov].sort_values(sensor_time).reset_index(drop=True)
        dov_df = dov_df.dropna(subset=[glucose])
        x = dov_df[glucose].interpolate(method="linear").values
        times = dov_df[sensor_time].values  # real datetimes for x-axis
        # Smooth
        fs = 1 / 300
        cutoff = 0.0005
        normalized_cutoff = cutoff / (fs / 2)
        b, a = butter(N=4, Wn=normalized_cutoff, btype="low")
        if smooth:
            x = filtfilt(b, a, x)
        # Sampling interval
        dt = np.median(np.diff(dov_df[sensor_time]))
        if hasattr(dt, "seconds"):
            dt = dt.seconds / 60
        samples_per_2h = int(time_between_peaks / dt)
        samples_per_30min = int(min_time_of_peak / dt)
        # Find peaks
        peaks, props = find_peaks(
            x,
            height=min_height,
            prominence=rise,
            distance=samples_per_2h,
            width=samples_per_30min,
            rel_height=1,
        )
        if clip:
            # Clip the peaks so the bases can't cross. For each peak and peak + 1,
            # trim the right edge of the peak and the left edge of the peak + 1
            for i in range(len(peaks) - 1):
                midpoint = (peaks[i] + peaks[i + 1]) / 2
                props["right_ips"][i] = min(props["right_ips"][i], midpoint)
                props["left_ips"][i + 1] = max(props["left_ips"][i + 1], midpoint)
        # Plot on this axis using real datetimes
        ax.plot(times, x, color="C0")
        ax.plot(times[peaks], x[peaks], "x", color="C1", markersize=10)
        ax.vlines(
            x=times[peaks],
            ymin=x[peaks] - props["prominences"],
            ymax=x[peaks],
            color="C1",
        )
        # width lines use interpolated indices — map back to datetime
        left_times = [times[int(i)] for i in props["left_ips"]]
        right_times = [times[int(i)] for i in props["right_ips"]]
        for lt, rt, wh in zip(left_times, right_times, props["width_heights"]):
            ax.hlines(y=wh, xmin=lt, xmax=rt, color="C1")
        ax.set_title(f"ID: {id}\nVisit: {pd.Timestamp(dov).date()}")
        ax.set_ylabel("Glucose (mg/dL)")
        ax.tick_params(axis="x", rotation=45)
        # Peak dataframe
        if len(peaks) > 0:
            peak_df = pd.DataFrame(
                {
                    "ID": id,
                    "dov_CGM": dov,
                    "left_base": dov_df[sensor_time]
                    .iloc[props["left_bases"].tolist()]
                    .values,
                    "left_ip": left_times,
                    "right_base": dov_df[sensor_time]
                    .iloc[props["right_bases"].tolist()]
                    .values,
                    "right_ip": right_times,
                    "peak_glucose": dov_df[glucose].iloc[peaks].tolist(),
                }
            )
            all_peak_dfs.append(peak_df)
    fig.tight_layout()
    plt.close(fig)
    all_peaks_df = (
        pd.concat(all_peak_dfs, ignore_index=True) if all_peak_dfs else pd.DataFrame()
    )
    return fig, all_peaks_df

In [ ]:
for i in cgm.ID.unique():
    fig, peak_df = plot_cgm_peaks(id=i)
    display(fig)
    display(peak_df)
    plt.close(fig)